# Silver Layer — Transaction Processing

## Objective
Transform raw banking transactions into validated, analytics-ready data while isolating invalid records.

## Implementation
- Process incremental transaction changes from Bronze using Delta Change Data Feed (CDF).
- Apply data validation, type conversions, and deduplication.
- Use Delta MERGE to maintain valid transactions in Silver.
- Route invalid transactions to a quarantine table for investigation.

## Validation
Successfully tested incremental ingestion with Batch 07:
- 3 transactions ingested into Bronze.
- 2 valid transactions processed in Silver.
- 1 invalid transaction captured in quarantine.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

bronze = "banking_project.bronze.bronze_transactions"
silver = "banking_project.silver.silver_transactions"
quarantine = "banking_project.silver.quarantine_transactions"

checkpoint = (
    "/Volumes/banking_project/silver/"
    "silver_pipeline_state/silver_cdf_checkpoint"
)

def process_transactions(batch_df, batch_id):

    if batch_df.isEmpty():
        return

    # 1. Keep inserts and updated records
    df = batch_df.filter(
        F.col("_change_type").isin("insert", "update_postimage")
    )

    # 2. Convert data types
    df = (
        df.withColumn(
            "transaction_date",
            F.expr("try_cast(transaction_date AS DATE)")
        )
        .withColumn(
            "transaction_amount",
            F.expr("try_cast(transaction_amount AS DECIMAL(18,2))")
        )
        .withColumn(
            "account_balance",
            F.expr("try_cast(account_balance AS DECIMAL(18,2))")
        )
        .withColumn(
            "emi_amount",
            F.expr("try_cast(emi_amount AS DECIMAL(18,2))")
        )
        .withColumn(
            "credit_score",
            F.expr("try_cast(credit_score AS INT)")
        )
        .withColumn(
            "transaction_hour",
            F.expr("try_cast(transaction_hour AS INT)")
        )
        .withColumn(
            "device_type",
            F.get_json_object("_rescued_data", "$.device_type")
        )
    )

    # 3. Validate transactions
    df = df.withColumn(
        "rejection_reason",
        F.concat_ws(
            "; ",
            F.when(
                F.col("transaction_id").isNull() |
                (F.trim("transaction_id") == ""),
                "Missing transaction ID"
            ),
            F.when(
                F.col("customer_id").isNull() |
                (F.trim("customer_id") == ""),
                "Missing customer ID"
            ),
            F.when(
                F.col("transaction_amount").isNull() |
                (F.col("transaction_amount") <= 0),
                "Invalid transaction amount"
            ),
            F.when(
                F.col("transaction_date").isNull(),
                "Invalid transaction date"
            ),
            F.when(
                F.col("account_balance").isNull(),
                "Invalid account balance"
            ),
            F.when(
                F.col("emi_amount").isNull(),
                "Invalid EMI amount"
            ),
            F.when(
                F.col("credit_score").isNotNull() &
                ~F.col("credit_score").between(300, 900),
                "Invalid credit score"
            ),
            F.when(
                F.col("transaction_hour").isNotNull() &
                ~F.col("transaction_hour").between(0, 23),
                "Invalid transaction hour"
            )
        )
    )

    # 4. Separate valid and invalid records
    valid = df.filter(F.col("rejection_reason") == "")
    invalid = df.filter(F.col("rejection_reason") != "")

    # 5. Quarantine invalid records
    # Reuse the existing quarantine schema and record hash
    raw_cols = [
        c for c in spark.table(bronze).columns
        if c not in ("ingestion_timestamp", "source_file")
    ]

    invalid = invalid.withColumn(
        "record_hash",
        F.sha2(
            F.to_json(
                F.struct(*[F.col(c) for c in raw_cols]),
                {"ignoreNullFields": "false"}
            ),
            256
        )
    )

    quarantine_schema = spark.table(quarantine).schema

    invalid = invalid.select(*[
        (
            F.col(field.name).cast(field.dataType)
            if field.name in invalid.columns
            else F.lit(None).cast(field.dataType)
        ).alias(field.name)
        for field in quarantine_schema
    ])

    invalid = invalid.dropDuplicates(["record_hash"])

    (
        DeltaTable.forName(spark, quarantine)
        .alias("target")
        .merge(
            invalid.alias("source"),
            "target.record_hash = source.record_hash"
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    # 6. Deduplicate valid transactions
    window = Window.partitionBy("transaction_id").orderBy(
        F.col("_commit_version").desc(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )

    valid = (
        valid.withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn", "rejection_reason")
    )

    # Match the existing Silver table schema
    silver_schema = spark.table(silver).schema

    valid = valid.select(*[
        F.col(field.name).cast(field.dataType).alias(field.name)
        for field in silver_schema
    ])

    # 7. MERGE valid records into Silver
    (
        DeltaTable.forName(spark, silver)
        .alias("target")
        .merge(
            valid.alias("source"),
            "target.transaction_id = source.transaction_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

# 8. Read CDF and process changes
query = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(bronze)
    .writeStream
    .foreachBatch(process_transactions)
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()

print("Silver Transactions CDF processing completed")

In [0]:
%sql
SELECT 'Silver Transactions' AS table_name, COUNT(*) AS records
FROM banking_project.silver.silver_transactions

UNION ALL

SELECT 'Quarantine Transactions', COUNT(*)
FROM banking_project.silver.quarantine_transactions;